Phase 2 - Reproduction of Garg & Soni (2025)

Module: Machine Learning and Related Applications  
Student: A. A. Nafais (KIC-BSCCSAI-251P-051)  
Paper: Garg, K., & Soni, U. (2025). Leveraging Unsupervised Machine Learning and Deep Learning for Enhanced Card Fraud Detection. Journal of Information Systems Engineering and Management, 10(4), 1927–1941.  
Dataset: card_transdata.csv - OpenML ID 45955.

Purpose
This notebook rebuilds the authors' pipeline exactly as the paper describes it, without any of my Phase 1 fixes. I've kept their flawed choices on purpose and marked them in the comments, so that:

1. my numbers can be compared with the numbers the authors reported, and
2. I can show that the improved notebook (phase 2 with solution.ipynb) fixes each problem.

The flawed choices I've kept from the paper: no EDA, MinMax scaling on all the features (even the binary ones), a single 80/20 split with no validation set, the paper's fixed settings (contamination = 0.02, nu = 0.02, autoencoder 16-8-16-7), thresholds picked by hand, and evaluation with accuracy and ROC-AUC only (no PR-AUC or MCC).


In [1]:
# setup, and a fixed random seed
import numpy as np, pandas as pd, random
np.random.seed(42); random.seed(42)

1. Load the dataset

In [2]:
from sklearn.datasets import fetch_openml
try:
    df = fetch_openml(data_id=45955, as_frame=True).frame.copy()
except Exception:
    df = pd.read_csv("card_transdata.csv")     # if OpenML fails, load the Kaggle copy
df["fraud"] = df["fraud"].astype(int)
print("Shape:", df.shape)
df.head()

/Users/amarnafais/Documents/nibm/degree/machine learning/phase 2/.venv/lib/python3.14/site-packages/sklearn/datasets/_openml.py:131: UserWarning: A network error occurred while downloading https://www.openml.org/api/v1/json/data/45955. Retrying...
  warn(


Shape: (1000000, 8)


,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0


2. Preprocessing (same as the paper)
There's no exploratory analysis here, because the paper didn't do any. MinMax scaling is applied to all seven features, including the four binary ones. This is what the paper did, and it's one of the problems I pointed out in Phase 1.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

features = ["distance_from_home","distance_from_last_transaction","ratio_to_median_purchase_price",
           "repeat_retailer","used_chip","used_pin_number","online_order"]
X, y = df[features], df["fraud"]

# same as paper: one 80/20 split and no validation set
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# same as paper (flawed): MinMax on all the features, even the binary ones
# I fit it on the training data only (the paper doesn't say how they did it)
scaler = MinMaxScaler().fit(X_tr)
X_trs = pd.DataFrame(scaler.transform(X_tr), columns=features, index=X_tr.index)
X_tes = pd.DataFrame(scaler.transform(X_te), columns=features, index=X_te.index)
print("train/test:", X_trs.shape[0], X_tes.shape[0])

train/test: 800000 200000


3. Evaluation helper (only the metrics the authors used)

In [4]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
def evaluate(name, y_true, y_pred, scores):
    # the paper only reports accuracy, precision, recall, F1 and ROC-AUC (no PR-AUC or MCC)
    return {"Model": name,
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, scores)}
results = []

4. The three unsupervised models (set up the same way as the paper)
All three models are trained on normal transactions only, and they use the settings given in the paper.

In [5]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

X_norm = X_trs[y_tr == 0]

# isolation forest: contamination is fixed at 0.02 like the paper
# this is far below the real fraud rate, so it limits recall (Phase 1, Proof 3)
iso = IsolationForest(n_estimators=100, contamination=0.02, random_state=42).fit(X_norm)
results.append(evaluate("Isolation Forest", y_te,
                        (iso.predict(X_tes) == -1).astype(int), -iso.score_samples(X_tes)))

# one-class svm: nu = 0.02 like the paper
# a kernel SVM is O(n^2), so training on all the normal rows would take far too long
# the paper doesn't mention this, so I train it on a sample of 20k normal rows to make it run
Xsub = X_norm.sample(min(20000, len(X_norm)), random_state=42)
oc = OneClassSVM(kernel="rbf", nu=0.02, gamma="scale").fit(Xsub)
results.append(evaluate("One-Class SVM", y_te,
                        (oc.predict(X_tes) == -1).astype(int), -oc.decision_function(X_tes)))
pd.DataFrame(results).round(3)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Isolation Forest,0.901,0.268,0.078,0.120,0.809
1,One-Class SVM,0.914,0.521,0.261,0.348,0.616


5. Autoencoder (set up the same way as the paper)
The layers are 16-8-16-7, trained for 50 epochs with Adam and MSE loss. A transaction is flagged as fraud when its reconstruction error is above the 95th percentile of the errors on the normal training rows. Like the paper, there's no validation set. This needs TensorFlow, which Colab already has.

In [6]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(42)

n = len(features)
Xn  = X_norm.values.astype("float32")
Xte = X_tes.values.astype("float32")

ae = keras.Sequential([
    layers.Input(shape=(n,)),
    layers.Dense(16, activation="relu"), layers.Dense(8, activation="relu"),
    layers.Dense(16, activation="relu"), layers.Dense(n, activation="linear"),
])
ae.compile(optimizer="adam", loss="mse")
ae.fit(Xn, Xn, epochs=50, batch_size=512, verbose=0)

err_tr = np.mean((Xn  - ae.predict(Xn,  verbose=0))**2, axis=1)   # same as paper: threshold comes from the normal training rows
err_te = np.mean((Xte - ae.predict(Xte, verbose=0))**2, axis=1)
thr = np.quantile(err_tr, 0.95)                                   # 95th percentile, same as the paper
results.append(evaluate("Autoencoder", y_te, (err_te > thr).astype(int), err_te))
pd.DataFrame(results).round(3)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Isolation Forest,0.901,0.268,0.078,0.120,0.809
1,One-Class SVM,0.914,0.521,0.261,0.348,0.616
2,Autoencoder,0.893,0.363,0.300,0.329,0.709


6. My reproduction vs the authors' reported results

In [7]:
ours = pd.DataFrame(results).round(3)
authors = pd.DataFrame([
    {"Model":"Authors — Isolation Forest","Accuracy":0.9023,"Precision":0.244,"Recall":0.056,"F1":0.091,"ROC-AUC":0.520},
    {"Model":"Authors — One-Class SVM",   "Accuracy":0.9279,"Precision":0.648,"Recall":0.382,"F1":0.481,"ROC-AUC":0.681},
    {"Model":"Authors — Autoencoder",     "Accuracy":0.9251,"Precision":0.560,"Recall":0.665,"F1":0.608,"ROC-AUC":0.807},
])
pd.concat([ours, authors], ignore_index=True).round(3)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Isolation Forest,0.901,0.268,0.078,0.120,0.809
1,One-Class SVM,0.914,0.521,0.261,0.348,0.616
2,Autoencoder,0.893,0.363,0.300,0.329,0.709
3,Authors — Isolation Forest,0.902,0.244,0.056,0.091,0.520
4,Authors — One-Class SVM,0.928,0.648,0.382,0.481,0.681
5,Authors — Autoencoder,0.925,0.560,0.665,0.608,0.807


7. Notes on the reproduction (fill in after running)

- My numbers are near / far (choose one) from the ones the authors reported. Some difference is expected, because the paper doesn't give the random seed, exactly how the train/test split was made, or the full autoencoder training details.
- This notebook keeps every flawed choice on purpose: no EDA, MinMax on the binary features, no validation set, thresholds set by hand, settings below the real fraud rate, and evaluation with only accuracy and ROC-AUC.
- The improved notebook (phase 2 with solution.ipynb) fixes these: it scales only the continuous features with RobustScaler, uses a train/validation/test split, sets the thresholds on validation data, sets the Isolation Forest contamination to the real fraud rate, adds a VAE, and uses PR-AUC and MCC because they give a fairer picture.